# NB04 — Too Many Features, and Regularization
**Machine Learning in Finance · Week 4**  [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ruiyuan-chen/ml-finance-course/blob/main/notebooks/NB04_regularized_regression.ipynb)

**Finance question:** A data vendor offers you dozens of new signals. Which ones actually matter?

**Key idea:** Every extra feature makes the model fit the **past** better, even a useless one. **Regularization** (Lasso, Ridge) charges the model for complexity, and Lasso switches useless features off completely.

In [ ]:
# ▶ RUN ME FIRST — loads the course tools and data (about 30 seconds the first time)
COURSE_URL = "https://raw.githubusercontent.com/ruiyuan-chen/ml-finance-course/main/"
import os, sys, urllib.request
here = [f for f in [".", "..", "../.."] if os.path.exists(os.path.join(f, "mlfin.py"))]
if here:
    sys.path.insert(0, here[0])
else:
    urllib.request.urlretrieve(COURSE_URL + "mlfin.py", "mlfin.py")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import mlfin
mlfin.setup(COURSE_URL)

## Step 1: the setup
We start where NB03 ended: a line of best guess for next month's return. This time we use **8 real
features** and train on just **three years (2015–2017)**, like a fund trying out a new data source
with a short history. The test period is still 2018–2025.

In [ ]:
panel = mlfin.make_features()
REAL = ["mom_12_1", "rev_1", "vol_12", "size", "btm", "ep", "beta_12", "max_1"]
for f in REAL:
    print(f"{f:10s} {mlfin.FEATURE_DESCRIPTIONS[f]}")

## Step 2: the vendor's "signals"
Imagine a vendor sells you extra signals with exciting names. Ours are **pure random numbers**: each
stock gets its own random level that wobbles a bit each month, so they *look* like real stock
characteristics, but by construction they know nothing about future returns.

In [ ]:
# 🟡 YOUR TURN
N_NOISE = 25   # ← how many useless signals to add; try 0, 50 or 100

data, NOISE = mlfin.add_noise_features(panel, N_NOISE)
train, test = mlfin.time_split(data, "2017-12-31")
train = train[train["date"] >= "2015-01-31"]          # three years of training data
data[["date", "ticker"] + REAL[:2] + NOISE[:3]].head()

## Step 3: fit the same line with and without the noise

In [ ]:
def evaluate(model, features, name):
    """Fit on the training period; report R-squared on both periods and the test top-minus-bottom spread."""
    X_tr, y_tr, _ = mlfin.prepare_xy(train, features)
    X_te, y_te, rows = mlfin.prepare_xy(test, features)
    model.fit(X_tr, y_tr)
    return pd.Series({"train R² (%)": model.score(X_tr, y_tr) * 100,
                      "test R² (%)": model.score(X_te, y_te) * 100,
                      "test top-minus-bottom (% a month)": mlfin.decile_spread(model.predict(X_te), y_te, rows["date"]) * 100},
                     name=name)

results = pd.DataFrame([
    evaluate(mlfin.linear_model("ols"), REAL, "8 real features"),
    evaluate(mlfin.linear_model("ols"), REAL + NOISE, f"8 real + {len(NOISE)} noise"),
]).round(3)
results

*What to look for:* with the noise added, the **training** R-squared goes **up** (the model found
"patterns" in random numbers) while the **test** R-squared goes **down**. `mlfin.linear_model` first puts every
feature on the same scale, so later we can compare the weights fairly.

## Step 4: the course's central picture
Now repeat the experiment for 0, 10, 25, 50 and 100 noise features. One line for the past (training),
one for the future (test). This takes about 20 seconds.

In [ ]:
# 🟡 YOUR TURN
SEED = 0   # ← a different seed draws different random noise. Is the picture just bad luck? Try 1 and 2.

curve = mlfin.noise_experiment(panel, REAL, noise_counts=[0, 10, 25, 50, 100], seed=SEED)
curve

*What to look for:* the navy line (training) climbs steadily: more features always make the past look
better. The gold line (test) drifts down. **Judging a model on the data it learned from rewards
complexity, not insight.** This is the same lesson as the Week 1 overfitting game, now with features.

## Step 5: Lasso — a model that pays for every feature
**Lasso** fits the same line but pays a price for every non-zero weight. Weak weights get pushed to
**exactly zero**: the feature is switched off. The **penalty** is the dial: 0 = plain regression; bigger =
fewer features survive.

In [ ]:
# 🟡 YOUR TURN
PENALTY = 0.001   # ← try 0.0005 (gentle), 0.002, 0.003, 0.005 (harsh)

lasso = mlfin.linear_model("lasso", PENALTY)
lasso_result = evaluate(lasso, REAL + NOISE, f"Lasso, penalty {PENALTY}")
weights = mlfin.plot_coefficients(lasso, REAL + NOISE)
lasso_result.round(3).to_frame()

*What to look for:* navy bars are real features, grey bars are noise. With a sensible penalty, the
noise disappears and only a few real features survive. Too harsh a penalty switches **everything**
off, and the model just predicts the average return.

## Step 6: Ridge — the other dial
**Ridge** also charges for complexity, but instead of switching features off it **shrinks every weight
toward zero**. Nothing is dropped. Ridge's penalty is on a different scale from Lasso's (bigger numbers).

In [ ]:
# 🟡 YOUR TURN
RIDGE_PENALTY = 10000   # ← try 1000 or 100000

ridge = mlfin.linear_model("ridge", RIDGE_PENALTY)
ridge_result = evaluate(ridge, REAL + NOISE, f"Ridge, penalty {RIDGE_PENALTY}")
_ = mlfin.plot_coefficients(ridge, REAL + NOISE)

In [ ]:
comparison = pd.DataFrame([results.iloc[1], ridge_result, lasso_result]).round(3)
comparison.index = ["Plain regression", "Ridge (shrinks all)", "Lasso (switches off)"]
comparison

*What to look for:* both penalized models do better on the **test** period than plain regression, even
though they fit the training data *less* well. Lasso has the bonus of a short, readable feature list:
you can explain it to an investment committee.

## Step 7: choose the penalty without peeking at the future
We must not pick the penalty by looking at the test period — that would be cheating. Instead we
cut the **training** period into blocks: learn on the early blocks, check on the next one, repeat.
This is **time-series cross-validation**: it never lets the model learn from the future.

In [ ]:
mlfin.plot_time_folds(train, n_splits=5)

In [ ]:
# 🟡 YOUR TURN
PENALTIES = [0, 0.0005, 0.001, 0.0015, 0.002, 0.003]   # ← the dial settings to try

cv_table = mlfin.tune_penalty(train, REAL + NOISE, PENALTIES, kind="lasso", n_splits=5)
cv_table

In [ ]:
best = cv_table["check-block spread (% a month)"].idxmax()
final = mlfin.linear_model("lasso", best)
final_result = evaluate(final, REAL + NOISE, f"Lasso, penalty chosen by CV = {best}")
_ = mlfin.plot_coefficients(final, REAL + NOISE)
final_result.round(3).to_frame()

*What to look for:* the penalty chosen inside the training period, then tested **once** on 2018–2025.
Notice the "worst block" column: even the best setting has bad stretches. Cross-validation reduces
the guesswork; it doesn't remove it.

## Step 8: your project's candidate features
Pick the real features your team is considering. Run Lasso with the chosen penalty and see which
survive. (Thursday's studio starts here.)

In [ ]:
# 🟡 YOUR TURN
MY_FEATURES = ["mom_12_1", "rev_1", "vol_12", "size", "btm", "ep", "beta_12", "max_1", "growth", "leverage"]
# ← add or remove from: mom_12_1, rev_1, vol_12, size, btm, ep, growth, margin, leverage, beta_12, max_1
MY_PENALTY = 0.001

mine = mlfin.linear_model("lasso", MY_PENALTY)
print(evaluate(mine, MY_FEATURES, "my features").round(3))
_ = mlfin.plot_coefficients(mine, MY_FEATURES)

## ⭐ GOING FURTHER (optional)
1. Ask an AI assistant: *"Why does Lasso set some weights to exactly zero while Ridge only shrinks them?
   Explain without equations."* Does its answer match what you saw in Steps 5–6?
2. In Step 2, change the training start from `"2015-01-31"` to `"2010-01-31"` (eight years). Does the noise
   still hurt as much? Why might more data protect you from overfitting?
3. Add `"sector"` to `MY_FEATURES` in Step 8. Which sectors survive?

## 🪞 REFLECTION (graded — write in your own words, no AI)

Double-click this cell and replace the *...* with your answers.

1. **In one sentence, what finance question did this week's technique answer?**
   *...*
2. **What is one way the result could be misleading?**
   *...*
3. **What did you ask an AI assistant this week, and was its answer right?**
   *...*